# 02b - Huấn luyện mô hình Cảm xúc (Fine-Tuning DistilBERT) trên Kaggle

**Mục tiêu:**
1. Load tập dữ liệu `IMDB_cleaned.csv` (đã được dọn dẹp sạch sẽ ở Local).
2. Mã hóa (Tokenize) văn bản để chuẩn bị cho mô hình Deep Learning.
3. Huấn luyện (Fine-tune) mô hình `distilbert-base-uncased` với cấu hình Log định kỳ.
4. Trích xuất lịch sử Log để vẽ biểu đồ **Loss (Độ mất mát)** và **Accuracy (Độ chính xác)**. Ngoài ra, tính thêm Confusion Matrix
5. Lưu mô hình (Model Weights) để tải về máy.


In [ ]:
# Chạy cell này để cài đặt thư viện cần thiết trên Kaggle
!pip install -q transformers datasets evaluate accelerate scikit-learn


## 1. Import Thư viện & Load Dữ Liệu
Hãy đảm bảo bạn đã **Add Data** file `IMDB_cleaned.csv` vào giao diện Kaggle. Bạn nhớ sửa lại đường dẫn `csv_path` bên dưới cho khớp với đường dẫn thực tế trên Kaggle nhé.


In [ ]:
import pandas as pd
import torch
import evaluate
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer, TrainingArguments
)

# 1. Load Data
# TODO: ĐỔI ĐƯỜNG DẪN NÀY THÀNH ĐƯỜNG DẪN FILE CSV TRÊN KAGGLE CỦA BẠN
csv_path = "/kaggle/input/your-dataset-name/IMDB_cleaned.csv" 
df = pd.read_csv(csv_path)

# Drop null phòng hờ
df = df.dropna(subset=["review", "label"])

# Chia tập Train (90%) và Val (10%)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])
print(f"Train size: {len(train_df)} | Validation size: {len(val_df)}")

# 2. Tokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    # Padding max_length=512 là giới hạn tối đa của BERT
    return tokenizer(batch["review"], truncation=True, max_length=512, padding="max_length")

print("Đang băm từ (Tokenizing) dữ liệu...")
train_ds = Dataset.from_pandas(train_df[["review","label"]]).map(tokenize_function, batched=True)
val_ds   = Dataset.from_pandas(val_df[["review","label"]]).map(tokenize_function, batched=True)


## 2. Cấu hình Huấn luyện & Ghi Log (Training Setup)
Để vẽ được biểu đồ sau này, ta cần định nghĩa hàm tính `Accuracy` và cài đặt tham số `logging_steps=500` (cứ học được 500 bước thì báo cáo kết quả 1 lần). Tránh việc log quá dày đặc gây tràn màn hình.


In [ ]:
# 1. Khởi tạo Mô hình
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 2. Hàm tính toán Accuracy (Dùng cho quá trình Validate)
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 3. Cấu hình Trainer
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=3,
    
    # 1. TĂNG BATCH SIZE TỐI ĐA (T4 có 16GB VRAM x 2 = 32GB VRAM)
    # Vì DistilBERT rất nhẹ, 1 GPU T4 có thể gánh được batch_size=32.
    # Với 2 GPU, tổng batch size thực tế mỗi bước học sẽ là 32 x 2 = 64! (Tốc độ x4 lần bình thường)
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    
    # 2. TĂNG SỐ LƯỢNG LUỒNG CPU ĐỌC DATA
    # Kaggle cấp cho bạn 4 nhân CPU. Cài = 2 để CPU đọc data kịp bơm cho 2 GPU cùng lúc, tránh bị nghẽn (bottleneck).
    dataloader_num_workers=2,
    
    evaluation_strategy="steps",
    eval_steps=500,
    logging_strategy="steps",            
    logging_steps=500,
    save_strategy="steps",
    save_steps=500,
    
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # 3. KÍCH HOẠT TENSOR CORES (Bắt buộc phải có)
    # Ép kiểu dữ liệu về Float 16. GPU T4 cực kỳ thích tính toán Float 16. Tốc độ train sẽ tăng 2-3 lần.
    fp16=True, 
    
    report_to="none"
)


#  Bắt đầu quá trình luyện đan
print(" BẮT ĐẦU HUẤN LUYỆN...")
trainer.train()


In [ ]:
# Code thêm vào Cell cuối cùng sau khi train xong
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

preds = trainer.predict(val_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = val_df["label"].tolist()

# In ra các chỉ số Precision, Recall, F1
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))

# Vẽ biểu đồ Ma trận
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["Negative", "Positive"], cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


## 3. Trực quan hóa Lịch sử Huấn luyện (Learning Curves)
Hugging Face Trainer tự động lưu toàn bộ log (Loss, Accuracy) vào `trainer.state.log_history`. Ta sẽ bóc tách dữ liệu này để vẽ biểu đồ chứng minh với nhà tuyển dụng rằng mô hình học rất tốt và không bị Overfitting.


In [ ]:
# Lấy lịch sử Log
log_history = trainer.state.log_history

train_steps, train_loss = [], []
eval_steps, eval_loss, eval_acc = [], [], []

# Tách riêng log của Train và Eval
for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    elif "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])
        eval_acc.append(log["eval_accuracy"])

# Vẽ biểu đồ Loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_steps, train_loss, label="Train Loss", marker='o')
plt.plot(eval_steps, eval_loss, label="Validation Loss", marker='s', color='red')
plt.title("Biểu đồ Độ mất mát (Loss)")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Vẽ biểu đồ Accuracy
plt.subplot(1, 2, 2)
plt.plot(eval_steps, eval_acc, label="Validation Accuracy", marker='s', color='green')
plt.title("Biểu đồ Độ chính xác (Accuracy)")
plt.xlabel("Training Steps")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## Santity check

In [ ]:
# Ví dụ: Câu này có chữ "good" nhưng thực ra là chê
test_sentence = "I really wanted to like this movie because the actors are good, but the plot is completely garbage."

inputs = tokenizer(test_sentence, return_tensors="pt").to("cuda") # Chạy trên GPU
outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=-1).item()

print("Cảm xúc dự đoán:", "Tích cực (Positive)" if prediction == 1 else "Tiêu cực (Negative)")
# Nếu AI đoán ra Negative, bạn đã thành công rực rỡ!


## 4. Lưu Mô hình (Save Model)


In [ ]:
save_path = "/kaggle/working/distilbert-imdb-sentiment"

# Lưu cả model và tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Hoàn tất! Model đã được lưu tại: {save_path}")
print("Hãy nhìn sang cột bên phải (Data -> Output) của Kaggle để tải thư mục này về máy tính nhé.")
